# SIMULACIONES – SOFTWARE FJ
**Sistema Integral de Gestión de Clientes, Servicios y Reservas**

Parte 5 del proyecto colaborativo · Programación (213023) · Grupo 213023_170 · UNAD – Fase 4

Este notebook pone a prueba todo el sistema con **15 simulaciones**, unas válidas y otras con errores provocados a propósito.
El objetivo es demostrar que, aunque se presenten errores graves, **el programa nunca se detiene**: cada error se captura, se registra en el archivo `logs/software_fj.log` y la ejecución continúa con la siguiente operación.

| Bloque | Simulaciones | Qué se prueba |
|---|---|---|
| A. Clientes | 1 a 3 | Registro válido e inválido (Parte 2) |
| B. Servicios | 4 a 10 | Creación correcta e incorrecta, polimorfismo y sobrecarga (Parte 3) |
| C. Reservas | 11 a 15 | Reservas exitosas y fallidas (Parte 4) |

**Cómo ejecutarlo:** abrir este archivo desde la carpeta raíz del repositorio (donde está la carpeta `software_fj`) y ejecutar las celdas en orden, de arriba hacia abajo.
En Google Colab, ejecutar primero en una celda nueva: `!git clone https://github.com/LeonardoGaratejo/software_fj.git` y luego `%cd software_fj`.

## 0. Preparación
Se importan las clases de las cinco partes y se define la función `simular`, que envuelve cada operación en `ejecutar_operacion` (Parte 1).
Esa función aplica `try / except / else / finally`, de modo que ningún error detiene el notebook.

In [13]:
# --- Importaciones: cada línea trae el aporte de una parte del proyecto ---
from software_fj.registro_log import ejecutar_operacion, ARCHIVO_LOG     # Parte 1: ejecución segura y ruta del log
from software_fj.cliente import Cliente                                  # Parte 2: clientes con datos validados
from software_fj.servicios import ReservaSala, AlquilerEquipo, Asesoria  # Parte 3: los tres servicios
from software_fj.reserva import Reserva                                  # Parte 4: reservas

resultados = []   # Aquí se guarda el resultado de cada simulación para el resumen final


def simular(numero, descripcion, funcion, *args, **kwargs):
    """Ejecuta una simulación de forma segura y muestra si terminó bien o con un error controlado."""
    resultado = ejecutar_operacion(f"Simulación {numero}: {descripcion}", funcion, *args, **kwargs)
    resultados.append((numero, descripcion, resultado))           # Se guarda para el resumen
    if resultado.exitosa:                                           # Camino normal
        print(f"✅ Simulación {numero} – ÉXITO: {resultado.valor}")
    else:                                                           # Camino del error: el programa sigue
        print(f"❌ Simulación {numero} – FALLO CONTROLADO: {type(resultado.error).__name__}: {resultado.error}")
    return resultado.valor                                          # Devuelve el objeto creado (o None si falló)


print("Sistema cargado. Log en:", ARCHIVO_LOG)

Sistema cargado. Log en: L:\UNAD\INGENIERIA DE SISTEMAS\SEMESTRE 03\PROGRAMACION\ETAPA 04\software_fj_repositorio\software_fj\logs\software_fj.log


## A. Clientes (Parte 2)
**Simulación 1** registra un cliente con datos correctos.
**Simulaciones 2 y 3** intentan registrar clientes con datos incorrectos: deben fallar de forma controlada.

In [14]:
# Simulación 1 (VÁLIDA): cliente con todos sus datos correctos
ana = simular(1, "Registrar cliente válido", Cliente,
              "C001", "Ana Pérez", "1075123456", "ana.perez@correo.com", "3101234567")

# Simulación 2 (INVÁLIDA): el correo no tiene @ -> error de dato inválido
simular(2, "Registrar cliente con correo inválido", Cliente,
        "C002", "Luis Rojas", "1075987654", "luis.correo.com", "3157654321")

# Simulación 3 (INVÁLIDA): el nombre está vacío -> ParametroFaltanteError
simular(3, "Registrar cliente sin nombre", Cliente,
        "C003", "", "1075111222", "sin.nombre@correo.com", "3001112233")

2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 1: Registrar cliente válido
2026-11-22 10:43:30 | INFO     | Cliente registrado: id=C001, nombre=Ana Pérez
2026-11-22 10:43:30 | INFO     | ÉXITO    | Simulación 1: Registrar cliente válido
2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 2: Registrar cliente con correo inválido
2026-11-22 10:43:30 | ERROR    | FALLO    | Simulación 2: Registrar cliente con correo inválido | DatoInvalidoError: [SFJ-100] Correo electrónico inválido: luis.correo.com
2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 3: Registrar cliente sin nombre
2026-11-22 10:43:30 | ERROR    | FALLO    | Simulación 3: Registrar cliente sin nombre | ParametroFaltanteError: [SFJ-200] El campo 'nombre' es obligatorio. (detalle: )


✅ Simulación 1 – ÉXITO: Cliente(id=C001, nombre=Ana Pérez, documento=1075123456, correo=ana.perez@correo.com, telefono=3101234567)
❌ Simulación 2 – FALLO CONTROLADO: DatoInvalidoError: [SFJ-100] Correo electrónico inválido: luis.correo.com
❌ Simulación 3 – FALLO CONTROLADO: ParametroFaltanteError: [SFJ-200] El campo 'nombre' es obligatorio. (detalle: )


## B. Servicios (Parte 3)
**Simulaciones 4 a 6** crean un servicio de cada tipo.
**Simulaciones 7 y 8** intentan crear servicios con datos incorrectos. La 7 muestra el **encadenamiento de excepciones**: en el log aparece el error del negocio y su CAUSA técnica.
**Simulación 9** demuestra el **polimorfismo** (la misma llamada en los tres servicios) y la **sobrecarga** de `calcular_costo` (sus tres formas).
**Simulación 10** provoca un **cálculo inconsistente**.

In [15]:
# Simulaciones 4 a 6 (VÁLIDAS): un servicio de cada tipo
sala = simular(4, "Crear sala de reuniones", ReservaSala, "S001", "Sala Huila", 50000, 25)
portatil = simular(5, "Crear alquiler de portátiles", AlquilerEquipo, "E001", "Portátil", 40000, 5)
asesoria = simular(6, "Crear asesoría en redes", Asesoria, "A001", "Asesoría en redes", 80000, "Redes", "senior")

# Simulación 7 (INVÁLIDA): tarifa con letras -> DatoInvalidoError encadenado a un ValueError
simular(7, "Crear sala con tarifa 'abc'", ReservaSala, "S002", "Sala Neiva", "abc", 10)

# Simulación 8 (INVÁLIDA): nivel de asesor que no existe
simular(8, "Crear asesoría con nivel inexistente", Asesoria, "A002", "Asesoría Python", 60000, "Python", "master")

2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 4: Crear sala de reuniones
2026-11-22 10:43:30 | INFO     | Servicio creado: Sala 'Sala Huila' para 25 personas, $50,000 por hora (disponible)
2026-11-22 10:43:30 | INFO     | ÉXITO    | Simulación 4: Crear sala de reuniones
2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 5: Crear alquiler de portátiles
2026-11-22 10:43:30 | INFO     | Servicio creado: Equipo 'Portátil' (5 unidades), $40,000 por día cada uno (disponible)
2026-11-22 10:43:30 | INFO     | ÉXITO    | Simulación 5: Crear alquiler de portátiles
2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 6: Crear asesoría en redes
2026-11-22 10:43:30 | INFO     | Servicio creado: Asesoría 'Asesoría en redes' en Redes, nivel senior, $80,000 por hora base (disponible)
2026-11-22 10:43:30 | INFO     | ÉXITO    | Simulación 6: Crear asesoría en redes
2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 7: Crear sala con tarifa 'abc'
2026-11-22 10:43:30 | ERROR    | FALLO

✅ Simulación 4 – ÉXITO: Sala 'Sala Huila' para 25 personas, $50,000 por hora (disponible)
✅ Simulación 5 – ÉXITO: Equipo 'Portátil' (5 unidades), $40,000 por día cada uno (disponible)
✅ Simulación 6 – ÉXITO: Asesoría 'Asesoría en redes' en Redes, nivel senior, $80,000 por hora base (disponible)
❌ Simulación 7 – FALLO CONTROLADO: DatoInvalidoError: [SFJ-100] El campo 'tarifa base' debe ser numérico. (detalle: abc)
❌ Simulación 8 – FALLO CONTROLADO: DatoInvalidoError: [SFJ-100] Nivel no válido. Opciones: basico, senior, experto. (detalle: master)


In [16]:
def comparar_costos():
    """POLIMORFISMO y SOBRECARGA en una sola operación."""
    costos = {}
    for servicio in (sala, portatil, asesoria):                   # La misma llamada para los tres...
        costos[servicio.nombre] = servicio.calcular_costo(2)      # ...y cada uno responde a su manera
    costos["Sala 3 h"] = sala.calcular_costo(3)                   # Firma 1: (duracion)
    costos["Sala 3 h + IVA 19 %"] = sala.calcular_costo(3, 19)    # Firma 2: (duracion, impuesto)
    costos["Sala 3 h + IVA 19 % - 10 %"] = sala.calcular_costo(3, 19, 10)  # Firma 3: (duracion, impuesto, descuento)
    return costos

# Simulación 9 (VÁLIDA): polimorfismo y sobrecarga de calcular_costo
costos = simular(9, "Comparar costos (polimorfismo y sobrecarga)", comparar_costos)
for concepto, valor in (costos or {}).items():                    # Tabla de resultados
    print(f"   {concepto:<30} $ {valor:>12,.0f}")

# Simulación 10 (INVÁLIDA): un descuento del 150 % no tiene sentido
simular(10, "Calcular costo con descuento de 150 %", sala.calcular_costo, 2, 19, 150)

2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 9: Comparar costos (polimorfismo y sobrecarga)
2026-11-22 10:43:30 | INFO     | calcular_costo(duracion) | Sala Huila | subtotal=120000.00 total=120000.00
2026-11-22 10:43:30 | INFO     | calcular_costo(duracion) | Portátil | subtotal=80000.00 total=80000.00
2026-11-22 10:43:30 | INFO     | calcular_costo(duracion) | Asesoría en redes | subtotal=240000.00 total=240000.00
2026-11-22 10:43:30 | INFO     | calcular_costo(duracion) | Sala Huila | subtotal=180000.00 total=180000.00
2026-11-22 10:43:30 | INFO     | calcular_costo(duracion, impuesto) | Sala Huila | subtotal=180000.00 total=214200.00
2026-11-22 10:43:30 | INFO     | calcular_costo(duracion, impuesto, descuento) | Sala Huila | subtotal=180000.00 total=192780.00
2026-11-22 10:43:30 | INFO     | ÉXITO    | Simulación 9: Comparar costos (polimorfismo y sobrecarga)
2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 10: Calcular costo con descuento de 150 %
2026-11-22 10:43:

✅ Simulación 9 – ÉXITO: {'Sala Huila': 120000.0, 'Portátil': 80000.0, 'Asesoría en redes': 240000.0, 'Sala 3 h': 180000.0, 'Sala 3 h + IVA 19 %': 214200.0, 'Sala 3 h + IVA 19 % - 10 %': 192780.0}
   Sala Huila                     $      120,000
   Portátil                       $       80,000
   Asesoría en redes              $      240,000
   Sala 3 h                       $      180,000
   Sala 3 h + IVA 19 %            $      214,200
   Sala 3 h + IVA 19 % - 10 %     $      192,780
❌ Simulación 10 – FALLO CONTROLADO: CalculoInconsistenteError: [SFJ-600] El descuento debe estar entre 0 % y 100 %. (detalle: 150)


## C. Reservas (Parte 4)
Cada reserva integra un cliente, un servicio y una duración, y avanza por estados: pendiente → confirmada → procesada (o cancelada).
**Simulación 11** es una reserva completa y exitosa.
**Simulaciones 12 a 15** son reservas fallidas: servicio no disponible, duración inválida, cliente inválido y una operación fuera de orden.
En las simulaciones 12 y 13 el log muestra el **encadenamiento**: el error de la reserva y, debajo, su CAUSA.

In [17]:
def reservar_y_procesar(id_reserva, cliente, servicio, duracion):
    """Recorrido completo de una reserva: crear, confirmar y procesar."""
    reserva = Reserva(id_reserva, cliente, servicio, duracion)   # Queda en estado pendiente
    reserva.confirmar()                                          # Pasa a confirmada
    reserva.procesar()                                           # Se procesa
    return reserva.describir()                                   # Resumen con el costo total


def cancelar_y_procesar(id_reserva, cliente, servicio, duracion):
    """Operación NO permitida: procesar una reserva que ya fue cancelada."""
    reserva = Reserva(id_reserva, cliente, servicio, duracion)
    reserva.cancelar("Prueba de operación no permitida")        # Se cancela...
    return reserva.procesar()                                    # ...y aun así se intenta procesar


# Simulación 11 (VÁLIDA): Ana reserva la Sala Huila por 3 horas
simular(11, "Reserva exitosa: Ana, Sala Huila, 3 horas", reservar_y_procesar, "R001", ana, sala, 3)

# Simulación 12 (INVÁLIDA): el servicio está en mantenimiento
portatil.deshabilitar()                                          # Se saca de uso a propósito
simular(12, "Reservar un equipo en mantenimiento", reservar_y_procesar, "R002", ana, portatil, 2)

# Simulación 13 (INVÁLIDA): una reserva de 0 horas
simular(13, "Reservar la sala por 0 horas", reservar_y_procesar, "R003", ana, sala, 0)

# Simulación 14 (INVÁLIDA): en lugar de un cliente se envía un texto
simular(14, "Reservar con un cliente inválido", reservar_y_procesar, "R004", "no soy un cliente", sala, 2)

# Simulación 15 (INVÁLIDA): procesar una reserva cancelada
simular(15, "Procesar una reserva cancelada", cancelar_y_procesar, "R005", ana, asesoria, 2)

2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 11: Reserva exitosa: Ana, Sala Huila, 3 horas
2026-11-22 10:43:30 | INFO     | calcular_costo(duracion) | Sala Huila | subtotal=180000.00 total=180000.00
2026-11-22 10:43:30 | INFO     | Reserva creada: R001 para cliente C001
2026-11-22 10:43:30 | INFO     | Reserva R001 confirmada
2026-11-22 10:43:30 | INFO     | Reserva R001 procesada
2026-11-22 10:43:30 | INFO     | ÉXITO    | Simulación 11: Reserva exitosa: Ana, Sala Huila, 3 horas
2026-11-22 10:43:30 | INFO     | Servicio deshabilitado: Portátil
2026-11-22 10:43:30 | INFO     | INICIO   | Simulación 12: Reservar un equipo en mantenimiento
2026-11-22 10:43:30 | ERROR    | FALLO    | Simulación 12: Reservar un equipo en mantenimiento | ReservaInvalidaError: [SFJ-400] Los datos de la reserva no son válidos.
2026-11-22 10:43:30 | ERROR    |   CAUSA  | ServicioNoDisponibleError: [SFJ-300] El servicio 'Portátil' no está disponible en este momento. (detalle: E001)
2026-11-22 10:43:30

✅ Simulación 11 – ÉXITO: Reserva R001: cliente=C001, servicio=Sala Huila, duracion=3.0, estado=confirmada, costo=180000.00
❌ Simulación 12 – FALLO CONTROLADO: ReservaInvalidaError: [SFJ-400] Los datos de la reserva no son válidos.
❌ Simulación 13 – FALLO CONTROLADO: ReservaInvalidaError: [SFJ-400] Los datos de la reserva no son válidos.
❌ Simulación 14 – FALLO CONTROLADO: AttributeError: 'str' object has no attribute 'id'
❌ Simulación 15 – FALLO CONTROLADO: OperacionNoPermitidaError: [SFJ-500] Solo se pueden procesar reservas confirmadas.


## D. Resumen y archivo de log
Si esta celda se ejecuta, quedó demostrado que el sistema **siguió funcionando** después de todos los errores anteriores.
Al final se muestran las últimas líneas del archivo de log, donde quedó registrado cada evento.

In [18]:
exitosas = [r for r in resultados if r[2].exitosa]          # Simulaciones que terminaron bien
fallidas = [r for r in resultados if not r[2].exitosa]       # Simulaciones con error controlado

print(f"Total de simulaciones: {len(resultados)}")
print(f"  Exitosas: {len(exitosas)}")
print(f"  Con error controlado: {len(fallidas)}")
print("El sistema siguió funcionando en todas ellas.\n")

for numero, descripcion, resultado in resultados:            # Tabla final
    estado = "ÉXITO" if resultado.exitosa else type(resultado.error).__name__
    print(f"{numero:>2}. {descripcion:<48} {estado}")

# Lectura del archivo de log ('with' cierra el archivo aunque ocurra un error)
print("\n--- Últimas líneas de", ARCHIVO_LOG.name, "---")
with open(ARCHIVO_LOG, encoding="utf-8") as archivo:
    for linea in archivo.readlines()[-12:]:
        print(linea.rstrip())

Total de simulaciones: 15
  Exitosas: 6
  Con error controlado: 9
El sistema siguió funcionando en todas ellas.

 1. Registrar cliente válido                         ÉXITO
 2. Registrar cliente con correo inválido            DatoInvalidoError
 3. Registrar cliente sin nombre                     ParametroFaltanteError
 4. Crear sala de reuniones                          ÉXITO
 5. Crear alquiler de portátiles                     ÉXITO
 6. Crear asesoría en redes                          ÉXITO
 7. Crear sala con tarifa 'abc'                      DatoInvalidoError
 8. Crear asesoría con nivel inexistente             DatoInvalidoError
 9. Comparar costos (polimorfismo y sobrecarga)      ÉXITO
10. Calcular costo con descuento de 150 %            CalculoInconsistenteError
11. Reserva exitosa: Ana, Sala Huila, 3 horas        ÉXITO
12. Reservar un equipo en mantenimiento              ReservaInvalidaError
13. Reservar la sala por 0 horas                     ReservaInvalidaError
14. Reservar con 